# RegNet-Y-3.2GF — From Scratch in PyTorch
**Paper:** Designing Network Design Spaces (Radosavovic et al., CVPR 2020)

| Config | Value |
|--------|-------|
| Series | RegNet Y |
| SE ratio | 0.25  (RegNet Y — Squeeze-and-Excitation) |
| depths | [2, 5, 13, 1] |
| widths | [72, 216, 576, 1512] |
| group_widths | [24, 24, 24, 24] |
| FC in_features | 1512 |
| Parameters | ~19.4M |
| Top-1 (ImageNet) | 78.9% |
| Input Size | 224x224 |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize

In [ ]:
# Cell 2 — RegNet-Y-3.2GF Architecture (from scratch)
import torch
import torch.nn as nn


class SE(nn.Module):
    def __init__(self, width, se_ratio=0.25):
        super().__init__()
        width_se = max(1, int(round(width * se_ratio)))
        self.excitation = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(width, width_se, 1, bias=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(width_se, width, 1, bias=True),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return x * self.excitation(x)


class XBlock(nn.Module):
    def __init__(self, width_in, width_out, stride, group_width,
                 bottleneck_multiplier=1.0, se_ratio=0.0):
        super().__init__()
        width_b = int(round(width_out * bottleneck_multiplier))
        groups  = max(1, width_b // group_width)
        layers  = [
            nn.Conv2d(width_in, width_b, 1, bias=False),
            nn.BatchNorm2d(width_b),
            nn.ReLU(inplace=True),
            nn.Conv2d(width_b, width_b, 3, stride=stride, padding=1,
                      groups=groups, bias=False),
            nn.BatchNorm2d(width_b),
            nn.ReLU(inplace=True),
        ]
        if se_ratio > 0.0:
            layers.append(SE(width_b, se_ratio))
        layers += [
            nn.Conv2d(width_b, width_out, 1, bias=False),
            nn.BatchNorm2d(width_out),
        ]
        self.f    = nn.Sequential(*layers)
        self.proj = nn.Sequential(
            nn.Conv2d(width_in, width_out, 1, stride=stride, bias=False),
            nn.BatchNorm2d(width_out),
        ) if (width_in != width_out or stride != 1) else nn.Identity()
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.f(x) + self.proj(x))


class RegNet(nn.Module):
    def __init__(self, stem_width, depths, widths, group_widths,
                 bottleneck_multiplier=1.0, se_ratio=0.0, num_classes=1000):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, stem_width, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(stem_width),
            nn.ReLU(inplace=True),
        )
        prev_w  = stem_width
        stages  = []
        for depth, width, gw in zip(depths, widths, group_widths):
            blocks = [XBlock(prev_w, width, stride=2, group_width=gw,
                             bottleneck_multiplier=bottleneck_multiplier,
                             se_ratio=se_ratio)]
            for _ in range(depth - 1):
                blocks.append(XBlock(width, width, stride=1, group_width=gw,
                                     bottleneck_multiplier=bottleneck_multiplier,
                                     se_ratio=se_ratio))
            stages.append(nn.Sequential(*blocks))
            prev_w = width
        self.trunk_output = nn.Sequential(*stages)
        self.avgpool      = nn.AdaptiveAvgPool2d(1)
        self.fc           = nn.Linear(prev_w, num_classes)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.stem(x)
        x = self.trunk_output(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)


# ── RegNet X (no SE) ──
def regnet_x_400mf(num_classes=1000):
    return RegNet(32, [1,2,7,12], [32,64,160,384],     [16,16,16,16],   se_ratio=0.0, num_classes=num_classes)

def regnet_x_800mf(num_classes=1000):
    return RegNet(32, [1,3,7,5],  [64,128,288,672],    [16,16,16,16],   se_ratio=0.0, num_classes=num_classes)

def regnet_x_1_6gf(num_classes=1000):
    return RegNet(32, [2,4,10,2], [72,168,408,912],    [24,24,24,24],   se_ratio=0.0, num_classes=num_classes)

def regnet_x_3_2gf(num_classes=1000):
    return RegNet(32, [2,6,15,2], [96,192,432,1008],   [48,48,48,48],   se_ratio=0.0, num_classes=num_classes)

def regnet_x_8gf(num_classes=1000):
    return RegNet(32, [2,5,15,1], [80,240,720,1920],   [120,120,120,120], se_ratio=0.0, num_classes=num_classes)

def regnet_x_16gf(num_classes=1000):
    return RegNet(32, [2,6,13,1], [256,512,896,2048],  [128,128,128,128], se_ratio=0.0, num_classes=num_classes)

def regnet_x_32gf(num_classes=1000):
    return RegNet(32, [2,7,13,1], [336,672,1344,2520], [168,168,168,168], se_ratio=0.0, num_classes=num_classes)


# ── RegNet Y (with SE, se_ratio=0.25) ──
def regnet_y_400mf(num_classes=1000):
    return RegNet(32, [1,3,6,6],  [48,104,208,440],    [8,8,8,8],       se_ratio=0.25, num_classes=num_classes)

def regnet_y_800mf(num_classes=1000):
    return RegNet(32, [1,3,8,2],  [64,128,320,768],    [16,16,16,16],   se_ratio=0.25, num_classes=num_classes)

def regnet_y_1_6gf(num_classes=1000):
    return RegNet(32, [2,6,17,2], [48,120,336,888],    [24,24,24,24],   se_ratio=0.25, num_classes=num_classes)

def regnet_y_3_2gf(num_classes=1000):
    return RegNet(32, [2,5,13,1], [72,216,576,1512],   [24,24,24,24],   se_ratio=0.25, num_classes=num_classes)

def regnet_y_8gf(num_classes=1000):
    return RegNet(32, [2,17,5,1], [168,448,896,2016],  [56,56,56,56],   se_ratio=0.25, num_classes=num_classes)

def regnet_y_16gf(num_classes=1000):
    return RegNet(32, [2,4,11,1], [224,448,1232,3024], [112,112,112,112], se_ratio=0.25, num_classes=num_classes)

def regnet_y_32gf(num_classes=1000):
    return RegNet(32, [2,5,12,1], [232,696,1392,3712], [232,232,232,232], se_ratio=0.25, num_classes=num_classes)

def regnet_y_128gf(num_classes=1000):
    return RegNet(32, [2,7,13,1], [528,1056,2904,7392],[264,264,264,264], se_ratio=0.25, num_classes=num_classes)


In [ ]:
NUM_CLASSES = 10
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')

model = regnet_y_3_2gf(num_classes=NUM_CLASSES).to(DEVICE)
print(model)

In [ ]:
dummy_input = torch.randn(2, 3, 224, 224).to(DEVICE)
output      = model(dummy_input)
print(f'Input  shape : {dummy_input.shape}')
print(f'Output shape : {output.shape}')

In [ ]:
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{'='*40}")
print(f"  Total parameters      : {total_params:,}")
print(f"  Trainable parameters  : {trainable_params:,}")
print(f"{'='*40}")

In [ ]:
print(f"{'Layer':<50} {'Shape':<28} {'Params':>10}")
print('-' * 90)
total = 0
for name, param in model.named_parameters():
    n = param.numel(); total += n
    print(f"{name:<50} {str(list(param.shape)):<28} {n:>10,}")
print('-' * 90)
print(f"{'TOTAL':<79} {total:>10,}")

---
## Training

In [ ]:
DATA_DIR   = './data'
BATCH_SIZE = 32

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
train_dataset = datasets.ImageFolder(f'{DATA_DIR}/train', transform=train_transform)
val_dataset   = datasets.ImageFolder(f'{DATA_DIR}/val',   transform=val_transform)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
CLASS_NAMES   = train_dataset.classes
print(f'Classes    : {CLASS_NAMES}')
print(f'Train size : {len(train_dataset)}')
print(f'Val size   : {len(val_dataset)}')

In [ ]:
EPOCHS = 20
LR     = 0.001

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

history = {'train_loss':[],'val_loss':[],'train_acc':[],'val_acc':[]}

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            if train: optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            if train: loss.backward(); optimizer.step()
            total_loss += loss.item() * images.size(0)
            correct    += outputs.max(1)[1].eq(labels).sum().item()
            total      += labels.size(0)
    return total_loss / total, 100.0 * correct / total

print(f"{'Epoch':<8} {'Tr Loss':<10} {'Tr Acc':<10} {'Val Loss':<10} {'Val Acc':<10}")
print('-' * 50)
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    vl_loss, vl_acc = run_epoch(val_loader,   train=False)
    scheduler.step()
    history['train_loss'].append(tr_loss); history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc);  history['val_acc'].append(vl_acc)
    print(f'{epoch:<8} {tr_loss:<10.4f} {tr_acc:<10.2f} {vl_loss:<10.4f} {vl_acc:<10.2f}')

torch.save(model.state_dict(), 'regnet_y_3_2gf_best.pth')
print('Model saved: regnet_y_3_2gf_best.pth')

---
## Training Curves

In [ ]:
epochs_range = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('RegNet-Y-3.2GF — Training Curves', fontsize=14, fontweight='bold')
axes[0].plot(epochs_range, history['train_loss'], 'b-o', linewidth=2, markersize=4, label='Train Loss')
axes[0].plot(epochs_range, history['val_loss'],   'r-o', linewidth=2, markersize=4, label='Val Loss')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(epochs_range, history['train_acc'], 'b-o', linewidth=2, markersize=4, label='Train Acc')
axes[1].plot(epochs_range, history['val_acc'],   'r-o', linewidth=2, markersize=4, label='Val Acc')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Inference

In [ ]:
from PIL import Image

def predict_image(image_path, top_k=5):
    model.eval()
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    img_pil = Image.open(image_path).convert('RGB')
    tensor  = transform(img_pil).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = F.softmax(model(tensor), dim=1)
    top_probs, top_idx = torch.topk(probs, top_k, dim=1)
    top_probs = top_probs[0].cpu().tolist()
    top_idx   = top_idx[0].cpu().tolist()
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].imshow(img_pil); axes[0].axis('off'); axes[0].set_title('Input')
    labels = [CLASS_NAMES[i] for i in top_idx]
    axes[1].barh(labels[::-1], [p*100 for p in top_probs[::-1]])
    axes[1].set_xlabel('Confidence (%)'); axes[1].set_title(f'Top-{top_k} Predictions')
    plt.tight_layout(); plt.show()
    print(f'Predicted: {CLASS_NAMES[top_idx[0]]}  ({top_probs[0]*100:.2f}%)')

# predict_image('your_image.jpg')
print('Call: predict_image("your_image.jpg")')

---
## ROC AUC Curve

In [ ]:
model.eval()
all_probs, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        probs = F.softmax(model(images.to(DEVICE)), dim=1)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.numpy())
all_probs  = np.concatenate(all_probs,  axis=0)
all_labels = np.concatenate(all_labels, axis=0)
print(f'Predictions shape : {all_probs.shape}')

In [ ]:
y_bin   = label_binarize(all_labels, classes=list(range(NUM_CLASSES)))
fig, ax = plt.subplots(figsize=(10, 8))
colors  = plt.cm.tab10(np.linspace(0, 1, NUM_CLASSES))
roc_auc_scores = {}
for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
    roc_auc     = auc(fpr, tpr)
    roc_auc_scores[CLASS_NAMES[i]] = roc_auc
    ax.plot(fpr, tpr, color=colors[i], linewidth=2,
            label=f'{CLASS_NAMES[i]}  (AUC = {roc_auc:.3f})')
macro_auc = roc_auc_score(y_bin, all_probs, average='macro', multi_class='ovr')
ax.plot([0,1],[0,1],'k--',linewidth=1,label='Random (AUC=0.500)')
ax.set_xlim([0,1]); ax.set_ylim([0,1.05])
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title(f'RegNet-Y-3.2GF — ROC AUC (Macro AUC = {macro_auc:.3f})', fontweight='bold')
ax.legend(loc='lower right', fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_auc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Macro AUC : {macro_auc:.4f}')